# AIBackends - native tool calling with LFM2.5-2.6B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-LFM2_5-tool-calling.ipynb)

LiquidAI's [LFM2.5-2.6B](https://huggingface.co/LiquidAI/LFM2.5-2.6B#tool-use) supports
native tool calling: tools are listed as JSON in the system prompt, and the model replies
with a Pythonic call such as `[get_weather(city="Paris")]` between tool-call tokens.
aibackends ships `extract_tool_calls` / `clean_answer` helpers to parse and strip them.
Mirrors `examples/tasks/tool_calling_lfm.py` on the llama.cpp runtime.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
import shutil
import subprocess

# Prebuilt llama-cpp-python wheels: CUDA 12.4 build on GPU runtimes, CPU build otherwise.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
LLAMA_WHEEL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35-cu124/llama_cpp_python-0.3.35-py3-none-manylinux_2_35_x86_64.whl"
    if HAS_NVIDIA_GPU
    else "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35/llama_cpp_python-0.3.35-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl"
)
print("GPU runtime:", HAS_NVIDIA_GPU)
print("llama-cpp-python wheel:", LLAMA_WHEEL.rsplit("/", 2)[-2])

%pip install -q "{LLAMA_WHEEL}"
%pip install -q "aibackends>=0.8.1" huggingface_hub

In [2]:
import shutil
import subprocess

import aibackends
import llama_cpp

# "gpu" offloads every layer to CUDA (n_gpu_layers=-1); "cpu" keeps everything on CPU.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
DEVICE = "gpu" if HAS_NVIDIA_GPU and llama_cpp.llama_supports_gpu_offload() else "cpu"

print("aibackends", aibackends.__version__)
print("llama-cpp-python", llama_cpp.__version__)
print("device:", DEVICE)

aibackends 0.8.1
llama-cpp-python 0.3.35
device: cpu


## 1. Define tools

Stub implementations so the notebook runs offline; swap in real APIs.

In [3]:
import json

TOOL_SCHEMAS = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g. Paris"}},
            "required": ["city"],
        },
    },
    {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Amount to convert"},
                "from_currency": {"type": "string", "description": "ISO code, e.g. USD"},
                "to_currency": {"type": "string", "description": "ISO code, e.g. EUR"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
]


def get_weather(city: str) -> dict:
    return {"city": city, "temperature_c": 21, "condition": "partly cloudy", "humidity": "58%"}


def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    rates = {("USD", "EUR"): 0.86, ("EUR", "USD"): 1.16, ("USD", "GBP"): 0.74}
    rate = rates.get((from_currency.upper(), to_currency.upper()))
    if rate is None:
        return {"error": f"Unsupported currency pair: {from_currency} to {to_currency}"}
    return {
        "amount": amount,
        "from_currency": from_currency.upper(),
        "to_currency": to_currency.upper(),
        "converted_amount": round(amount * rate, 2),
        "rate": rate,
    }


TOOL_FUNCTIONS = {"get_weather": get_weather, "convert_currency": convert_currency}

## 2. Load LFM2.5-2.6B

`LFM25_2_6B` resolves to the Q4_K_M GGUF from `LiquidAI/LFM2.5-2.6B-GGUF` (~1.6 GB).
Try `quantization="Q8_0"` for higher fidelity. `skip_special_tokens=False` keeps the
tool-call markers in the output so they can be parsed.

In [4]:
from aibackends import get_runtime
from aibackends.models import LFM25_2_6B
from aibackends.runtimes import LLAMACPP

runtime = get_runtime({
    "runtime": LLAMACPP,
    "model": LFM25_2_6B,
    "device": DEVICE,
    "max_tokens": 1024,
    "extra_options": {"skip_special_tokens": False},
})

## 3. The tool loop

Model picks a tool -> we execute it -> the result goes back with the `tool` role -> the model answers.

In [5]:
from aibackends.core.tool_calls import clean_answer, extract_tool_calls


def run_tool_loop(question: str, max_rounds: int = 5) -> str:
    messages = [
        {"role": "system", "content": f"List of tools: {json.dumps(TOOL_SCHEMAS)}"},
        {"role": "user", "content": question},
    ]
    print(f"[user] {question}")
    for round_number in range(max_rounds):
        response = runtime.complete(messages)
        tool_calls = extract_tool_calls(response.content)
        if not tool_calls:
            answer = clean_answer(response.content)
            label = "final answer" if round_number else "model, no tools"
            print(f"[{label}] {answer}\n")
            return answer

        results = []
        for call in tool_calls:
            print(f"[tool call] {call.name}({call.arguments})")
            function = TOOL_FUNCTIONS.get(call.name)
            result = function(**call.arguments) if function else {"error": f"Unknown tool: {call.name}"}
            print(f"[tool result] {json.dumps(result)}")
            results.append(result)

        messages.append({"role": "assistant", "content": clean_answer(response.content)})
        messages.append({"role": "tool", "content": json.dumps(results)})
    raise RuntimeError(f"No final answer after {max_rounds} tool-calling rounds")


run_tool_loop("What's the weather like in Paris right now?")
run_tool_loop("How much is 250 USD in EUR?")
run_tool_loop("Tell me a one-line joke about compilers.")

[user] What's the weather like in Paris right now?


[tool call] get_weather({'city': 'Paris'})
[tool result] {"city": "Paris", "temperature_c": 21, "condition": "partly cloudy", "humidity": "58%"}


[final answer] The current weather in Paris is **partly cloudy** with a temperature of **21°C** and humidity at **58%**.

[user] How much is 250 USD in EUR?


[tool call] convert_currency({'amount': 250, 'from_currency': 'USD', 'to_currency': 'EUR'})
[tool result] {"amount": 250, "from_currency": "USD", "to_currency": "EUR", "converted_amount": 215.0, "rate": 0.86}


[final answer] 250 USD is equal to **215.00 EUR** (exchange rate: 0.86).

[user] Tell me a one-line joke about compilers.


[model, no tools] Why did the compiler break up with the programmer? It couldn't handle their incompatible syntax!



"Why did the compiler break up with the programmer? It couldn't handle their incompatible syntax!"